### 1. 카드사의 VOC를 활용하여, 로컬에서 RAG를 수행하는 streamlit을 만들어보세요.

- 다운로드 및 처리

In [ ]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/voc_card.zip

In [ ]:
!unzip -O CP949 voc_card.zip -d ./voc_card/

In [ ]:
import json
import glob

# 모든 json 파일 읽기
json_files = glob.glob('./voc_card/*.json')

data_list = []
for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        data_list.append(data[0])

print(f"총 {len(data_list)}개 파일 로드")
print(data_list[0])  # 첫 번째 확인

- Chroma DB RAG 설정

In [ ]:
!pip install -q \
  opentelemetry-sdk==1.27.0 \
  opentelemetry-api==1.27.0 \
  opentelemetry-exporter-otlp-proto-grpc==1.27.0 \
  chromadb \
  langchain-chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.27.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.27.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.27.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but 

In [ ]:
# chroma_db 폴더 생성

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings

vectorstore = Chroma(
    collection_name="factory_docs",
    embedding_function=OllamaEmbeddings(model="nomic-embed-text"),
    persist_directory="./chroma_db"  # 디스크에 저장
)

# 문서 추가
#vectorstore.add_documents(문자열 리스트, ids=[문서의 개수와 동일한 아이디 문자열])

/tmp/ipykernel_28189/3124549173.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings
/tmp/ipykernel_28189/3124549173.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding_function=OllamaEmbeddings(model="nomic-embed-text"),


- Chroma DB에 문서 추가

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.documents import Document          
from langchain_classic.chains import RetrievalQA       
import json

In [ ]:
# --- reports.json 로드 ---
reports = reports[:10]

#lc_embeddings = OllamaEmbeddings(model="nomic-embed-text")

lc_documents = [
    Document(
        page_content=r["report_text"],
        metadata={"report_id": r["report_id"], "source": "json"}
    )
    for r in reports
]

In [ ]:
#Chroma DB에 추가
vectorstore.add_documents(lc_documents)
print(f"추가 완료: 총 {vectorstore._collection.count()}개")

추가 완료: 총 10개


- Vector Store 검색 tool 정의

In [ ]:
# 벡터스토어 검색 리트리버 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

@tool
def search_engine_reports(query: str) -> str:
    """엔진 고장 및 불량 보고서에서 관련 정보를 검색합니다. 제조 불량 원인 질문에 사용하세요."""
    docs = retriever.invoke(query)
    if not docs:
        return "관련 문서를 찾지 못했습니다."

    results = []
    for i, doc in enumerate(docs, 1):
        src    = doc.metadata.get("source", "unknown")
        ref_id = doc.metadata.get("report_id", doc.metadata.get("page", ""))
        snippet = doc.page_content[:300]
        results.append(f"[{i}] 출처={src} | ID/페이지={ref_id}\n{snippet}")

    return "\n\n".join(results)

tools.append(search_engine_reports)

- streamlit 부분

In [ ]:
import streamlit as st
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

st.title("엔진 진단 챗봇")

# 모델 초기화 (세션당 1회)
if "llm" not in st.session_state:
    st.session_state.llm = ChatOllama(
        model="qwen2.5:3b",
        temperature=0.2,
        timeout=120,
    )

# 대화 기록 초기화
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 대화 출력
for msg in st.session_state.messages:
    role = "user" if isinstance(msg, HumanMessage) else "assistant"
    with st.chat_message(role):
        st.write(msg.content)

# 입력
#prompt = st.chat_input("질문")
#if prompt:
#    ...
if prompt := st.chat_input("질문을 입력하세요"):        #월루스 연산자, 할당 표현식
    # 사용자 메시지 추가 및 출력
    st.session_state.messages.append(HumanMessage(content=prompt))
    with st.chat_message("user"):
        st.write(prompt)

    # LLM 호출 (전체 대화 기록 전달 → 멀티턴)
    with st.chat_message("assistant"):
        with st.spinner("생각 중..."):
            #tmp = st.session_state.llm.bind_tools(  )
            response = tmp.invoke(st.session_state.messages)
        st.write(response.content)

    # 응답 저장
    st.session_state.messages.append(AIMessage(content=response.content))